In [2]:
%pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import mysql.connector
from getpass import getpass

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass("Enter MySQL password:"),
    database="taxix"
)

if conn.is_connected():
    print("MySQL connected successfully!")

Enter MySQL password: ········


MySQL connected successfully!


In [5]:
import os
import csv
import mysql.connector

folder_path = r"C:\Users\HP\OneDrive\Desktop\WS CUBE DATA\Capstone_project\Uber_Data\taxi_trips"

file_table_map = {
    "2017_taxi_trips.csv": "stg_uber_2017",
    "2018_taxi_trips.csv": "stg_uber_2018",
    "2019_taxi_trips.csv": "stg_uber_2019",
    "2020_taxi_trips.csv": "stg_uber_2020"
}

cursor = conn.cursor()

for filename, table_name in file_table_map.items():

    file_path = os.path.join(folder_path, filename)

    if not os.path.exists(file_path):
        print("File not found:", file_path)
        continue

    print(f"\nImporting {filename}...")

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        reader = csv.reader(f)
        headers = next(reader)

        # Create a staging table with text columns
        # to avoid guessing CSV data types.
        cursor.execute(f"DROP TABLE IF EXISTS `{table_name}`")

        columns_sql = ", ".join(
            f"`{col.replace('`', '``')}` LONGTEXT"
            for col in headers
        )

        cursor.execute(
            f"CREATE TABLE `{table_name}` ({columns_sql})"
        )
        conn.commit()

        column_names = ", ".join(
            f"`{col.replace('`', '``')}`"
            for col in headers
        )

        placeholders = ", ".join(["%s"] * len(headers))

        insert_sql = (
            f"INSERT INTO `{table_name}` "
            f"({column_names}) VALUES ({placeholders})"
        )

        batch = []
        imported_rows = 0

        for row in reader:

            if len(row) != len(headers):
                raise ValueError(
                    f"Column mismatch in {filename}, "
                    f"row {imported_rows + 1}"
                )

            batch.append(tuple(
                value if value != "" else None
                for value in row
            ))

            if len(batch) >= 2000:
                cursor.executemany(insert_sql, batch)
                conn.commit()

                imported_rows += len(batch)
                batch.clear()

                print(f"Imported {imported_rows:,} rows")

        if batch:
            cursor.executemany(insert_sql, batch)
            conn.commit()
            imported_rows += len(batch)

    cursor.execute(f"SELECT COUNT(*) FROM `{table_name}`")
    db_count = cursor.fetchone()[0]

    print("CSV rows imported:", imported_rows)
    print("MySQL rows:", db_count)

    if imported_rows != db_count:
        print("WARNING: Row counts do not match!")
    else:
        print("Row count verified successfully!")


Importing 2017_taxi_trips.csv...
Imported 2,000 rows
Imported 4,000 rows
Imported 6,000 rows
Imported 8,000 rows
Imported 10,000 rows
Imported 12,000 rows
Imported 14,000 rows
Imported 16,000 rows
Imported 18,000 rows
Imported 20,000 rows
Imported 22,000 rows
Imported 24,000 rows
Imported 26,000 rows
Imported 28,000 rows
Imported 30,000 rows
Imported 32,000 rows
Imported 34,000 rows
Imported 36,000 rows
Imported 38,000 rows
Imported 40,000 rows
Imported 42,000 rows
Imported 44,000 rows
Imported 46,000 rows
Imported 48,000 rows
Imported 50,000 rows
Imported 52,000 rows
Imported 54,000 rows
Imported 56,000 rows
Imported 58,000 rows
Imported 60,000 rows
Imported 62,000 rows
Imported 64,000 rows
Imported 66,000 rows
Imported 68,000 rows
Imported 70,000 rows
Imported 72,000 rows
Imported 74,000 rows
Imported 76,000 rows
Imported 78,000 rows
Imported 80,000 rows
Imported 82,000 rows
Imported 84,000 rows
Imported 86,000 rows
Imported 88,000 rows
Imported 90,000 rows
Imported 92,000 rows
Impo

In [6]:
cursor = conn.cursor()

for year in [2017, 2018, 2019, 2020]:
    table_name = f"stg_uber_{year}"

    cursor.execute(f"SELECT COUNT(*) FROM `{table_name}`")
    count = cursor.fetchone()[0]

    print(f"{year}: {count:,} rows")

2017: 1,048,575 rows
2018: 8,807,303 rows
2019: 6,044,050 rows
2020: 1,734,051 rows


In [7]:
import os
import csv

folder_path = r"C:\Users\HP\OneDrive\Desktop\WS CUBE DATA\Capstone_project\Uber_Data\taxi_trips"

file_table_map = {
    "2017_taxi_trips.csv": "stg_uber_2017",
    "2018_taxi_trips.csv": "stg_uber_2018",
    "2019_taxi_trips.csv": "stg_uber_2019",
    "2020_taxi_trips.csv": "stg_uber_2020"
}

for filename, table_name in file_table_map.items():
    path = os.path.join(folder_path, filename)

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        csv_count = sum(1 for _ in csv.reader(f)) - 1

    cursor.execute(f"SELECT COUNT(*) FROM `{table_name}`")
    mysql_count = cursor.fetchone()[0]

    print(f"\n{filename}")
    print(f"CSV rows:   {csv_count:,}")
    print(f"MySQL rows: {mysql_count:,}")
    print("MATCH" if csv_count == mysql_count else "MISMATCH")


2017_taxi_trips.csv
CSV rows:   1,048,575
MySQL rows: 1,048,575
MATCH

2018_taxi_trips.csv
CSV rows:   8,807,303
MySQL rows: 8,807,303
MATCH

2019_taxi_trips.csv
CSV rows:   6,044,050
MySQL rows: 6,044,050
MATCH

2020_taxi_trips.csv
CSV rows:   1,734,051
MySQL rows: 1,734,051
MATCH


In [8]:
import os
import csv

folder_path = r"C:\Users\HP\OneDrive\Desktop\WS CUBE DATA\Capstone_project\Uber_Data\taxi_trips"

file_table_map = {
    "2017_taxi_trips.csv": "stg_uber_2017",
    "2018_taxi_trips.csv": "stg_uber_2018",
    "2019_taxi_trips.csv": "stg_uber_2019",
    "2020_taxi_trips.csv": "stg_uber_2020"
}

all_match = True

for filename, table_name in file_table_map.items():
    path = os.path.join(folder_path, filename)

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        csv_count = sum(1 for _ in csv.reader(f)) - 1

    cursor.execute(f"SELECT COUNT(*) FROM `{table_name}`")
    mysql_count = cursor.fetchone()[0]

    status = "MATCH" if csv_count == mysql_count else "MISMATCH"
    print(f"{filename}: CSV={csv_count:,}, MySQL={mysql_count:,} -> {status}")

    if csv_count != mysql_count:
        all_match = False

print("\nAll years verified:", all_match)

2017_taxi_trips.csv: CSV=1,048,575, MySQL=1,048,575 -> MATCH
2018_taxi_trips.csv: CSV=8,807,303, MySQL=8,807,303 -> MATCH
2019_taxi_trips.csv: CSV=6,044,050, MySQL=6,044,050 -> MATCH
2020_taxi_trips.csv: CSV=1,734,051, MySQL=1,734,051 -> MATCH

All years verified: True


In [16]:
cursor = conn.cursor()

cursor.execute("SHOW TABLES FROM taxix")

tables = cursor.fetchall()

for table in tables:
    print(table[0])

stg_uber_2017
stg_uber_2018
stg_uber_2019
stg_uber_2020
